# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [13]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [5]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "../ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()
print(len(docs))

26


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from pydantic import BaseModel
from openai import OpenAI
import os

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

selected_doc = docs[0]
article_text = selected_doc.page_content

tone = "Victorian English"

instructions = f"""You are an expert summarizer. Your task is to analyze the provided article and produce a structured summary with the following fields:
- Author: The author of the article, if mentioned; otherwise, infer or state 'Unknown'.
- Title: The title of the article, if mentioned; otherwise, infer or state 'Unknown'.
- Relevance: A statement, no longer than one paragraph, explaining why this article is relevant for an AI professional in their professional development.
- Summary: A concise and succinct summary, no longer than 1000 tokens, written in {tone} style.
- Tone: The tone used, which is '{tone}'.
- InputTokens: Will be filled from usage data.
- OutputTokens: Will be filled from usage data.

Ensure the summary captures the key points of the article while adhering strictly to the {tone} style."""

user_prompt = f"Article content:\n{article_text}"

response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",  
    messages=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ],
    response_format=ArticleSummary
)


parsed_summary = response.choices[0].message.parsed


parsed_summary.InputTokens = response.usage.prompt_tokens
parsed_summary.OutputTokens = response.usage.completion_tokens


print(parsed_summary)

Author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari' Title='The GenAI Divide: State of AI in Business 2025' Relevance='This article is of significant relevance to AI professionals as it delineates the current trends and forecasted trajectories of artificial intelligence within the realm of business, thereby equipping practitioners with an understanding of the evolving landscape and implications for future developments in their field.' Summary="In the year of our Lord 2025, the illustrious treatise entitled 'The GenAI Divide' hath been penned by noteworthy scholars, including the esteemed MIT Nanda. The work posits a thorough examination of the integration and impact of artificial intelligence within the business sector. This document elucidates the dichotomy extant among enterprises as they embrace generative AI, revealing a chasm that separates the efficacious from the laggards in the field. Furthermore, the authors articulate the manifold opportunities arising fro

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
from openai import OpenAI
from pydantic import BaseModel
import os
from deepeval.models.base_model import DeepEvalBaseLLM

class GatewayModel(DeepEvalBaseLLM):
    def __init__(self, client, model):
        self._client = client
        self._model = model

    def load_model(self):
        return self._client

    def generate(self, prompt: str) -> str:
        response = self._client.chat.completions.create(
            model=self._model,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return self._model

article_text = docs[0].page_content

if 'parsed_summary' not in locals():
    class ArticleSummary(BaseModel):
        Author: str
        Title: str
        Relevance: str
        Summary: str
        Tone: str
        InputTokens: int
        OutputTokens: int
    
    tone = "Victorian English"
    instructions = f"""You are an expert summarizer. Your task is to analyze the provided article and produce a structured summary with the following fields:
- Author: The author of the article, if mentioned; otherwise, infer or state 'Unknown'.
- Title: The title of the article, if mentioned; otherwise, infer or state 'Unknown'.
- Relevance: A statement, no longer than one paragraph, explaining why this article is relevant for an AI professional in their professional development.
- Summary: A concise and succinct summary, no longer than 1000 tokens, written in {tone} style.
- Tone: The tone used, which is '{tone}'.
- InputTokens: Will be filled from usage data.
- OutputTokens: Will be filled from usage data.

Ensure the summary captures the key points of the article while adhering strictly to the {tone} style."""

    user_prompt = f"Article content:\n{article_text}"
    
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",  
        messages=[
            {"role": "developer", "content": instructions},
            {"role": "user", "content": user_prompt}
        ],
        response_format=ArticleSummary
    )
    
    parsed_summary = response.choices[0].message.parsed
    parsed_summary.InputTokens = response.usage.prompt_tokens
    parsed_summary.OutputTokens = response.usage.completion_tokens

gateway_base_url = 'https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1'
gateway_api_key = os.getenv('API_GATEWAY_KEY')

# Create AI judge model using the gateway
judge_model = GatewayModel(client=client, model="gpt-4o-mini")


print("=" * 70)
print("EVALUATING SUMMARY WITH DEEPEVAL METRICS")
print("=" * 70)

# ==============================================================================
# 1. SUMMARIZATION METRIC
# ==============================================================================
print("\n[1/4] Starting Summarization Evaluation...")

summarization_questions = [
    "Does the summary accurately capture the main topic of the original article?",
    "Are all key points from the original article included in the summary?",
    "Is the summary concise and avoid unnecessary details?",
    "Does the summary maintain the factual accuracy of the original text?",
    "Is the summary well-organized and logically structured?"
]

summarization_metric = SummarizationMetric(
    assessment_questions=summarization_questions,
    model=judge_model,
    async_mode=False
)

test_case = LLMTestCase(
    input=article_text,
    actual_output=parsed_summary.Summary
)

summarization_score = summarization_metric.measure(test_case)
def extract_score_and_reason(result):
    if isinstance(result, (int, float)):
        return float(result), ""
    return getattr(result, "score", None), getattr(result, "reason", "")

score, reason = extract_score_and_reason(summarization_score)

if isinstance(score, (int, float)):
    print(f"✓ Summarization Score: {score:.2f}")
else:
    print(f"✗ Summarization Score unavailable: {score}")
# ==============================================================================
# 2. COHERENCE METRIC (G-Eval)
# ==============================================================================
print("\n[2/4] Starting Coherence Evaluation...")

coherence_questions = [
    "Is the summary written in clear, understandable language?",
    "Are the ideas presented in a logical sequence?",
    "Does each sentence flow naturally to the next?",
    "Are technical terms explained or used appropriately?",
    "Is the overall message easily understood on first reading?"
]

coherence_criteria = "Evaluate the clarity and coherence of the summary. The summary should be written in clear, understandable language with ideas presented in logical sequence."

coherence_metric = GEval(
    name="Coherence",
    criteria=coherence_criteria,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=coherence_questions,
    model=judge_model,
    async_mode=False
)

coherence_score = coherence_metric.measure(test_case)
score, reason = extract_score_and_reason(coherence_score)

if isinstance(score, (int, float)):
    print(f"✓ Coherence Score: {score:.2f}")
else:
    print(f"✗ Coherence Score unavailable: {score}")
# ==============================================================================
# 3. TONALITY METRIC (G-Eval)
# ==============================================================================
print("\n[3/4] Starting Tonality Evaluation...")

tonality_questions = [
    f"Does the summary consistently maintain {parsed_summary.Tone} tone?",
    f"Are the stylistic choices appropriate for {parsed_summary.Tone}?",
    "Is the tone maintained throughout without jarring shifts?",
    "Does the tone enhance or detract from the message?",
    "Would a reader recognize this as written in the intended tone?"
]

tonality_criteria = f"Evaluate if the summary maintains the {parsed_summary.Tone} tone consistently throughout. The tone should be distinguishable and appropriate for the content."

tonality_metric = GEval(
    name="Tonality",
    criteria=tonality_criteria,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=tonality_questions,
    model=judge_model,
    async_mode=False
)

tonality_score = tonality_metric.measure(test_case)
score, reason = extract_score_and_reason(tonality_score)

if isinstance(score, (int, float)):
    print(f"✓ Tonality Score: {score:.2f}")
else:
    print(f"✗ Tonality Score unavailable: {score}")

# ==============================================================================
# 4. SAFETY METRIC (G-Eval)
# ==============================================================================
print("\n[4/4] Starting Safety Evaluation...")

safety_questions = [
    "Does the summary avoid making harmful or misleading claims?",
    "Are claims properly attributed or qualified?",
    "Does the summary avoid biased or discriminatory language?",
    "Are conclusions supported by evidence from the original text?",
    "Does the summary avoid sensitive information inappropriately?"
]

safety_criteria = "Evaluate the safety and integrity of the summary. It should avoid harmful claims, maintain proper attribution, use unbiased language, and handle sensitive information appropriately."

safety_metric = GEval(
    name="Safety",
    criteria=safety_criteria,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=safety_questions,
    model=judge_model,
    async_mode=False
)

safety_score = safety_metric.measure(test_case)
score, reason = extract_score_and_reason(safety_score)

if isinstance(score, (int, float)):
    print(f"✓ Safety Score: {score:.2f}")
else:
    print(f"✗ Safety Score unavailable: {score}")

# ==============================================================================
# STRUCTURE OUTPUT RESULTS
# ==============================================================================
print("\n" + "=" * 70)
print("EVALUATION RESULTS")
print("=" * 70)

class EvaluationResults(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

summ_score, summ_reason = extract_score_and_reason(summarization_score)
coh_score, coh_reason = extract_score_and_reason(coherence_score)
ton_score, ton_reason = extract_score_and_reason(tonality_score)
safe_score, safe_reason = extract_score_and_reason(safety_score)

evaluation_results = EvaluationResults(
    SummarizationScore=summ_score,
    SummarizationReason=summ_reason or "",
    CoherenceScore=coh_score,
    CoherenceReason=coh_reason or "",
    TonalityScore=ton_score,
    TonalityReason=ton_reason or "",
    SafetyScore=safe_score,
    SafetyReason=safe_reason or ""
)


print(evaluation_results.model_dump_json(indent=2))

EVALUATING SUMMARY WITH DEEPEVAL METRICS

[1/4] Starting Summarization Evaluation...


✓ Summarization Score: 0.00

[2/4] Starting Coherence Evaluation...


✓ Coherence Score: 0.60

[3/4] Starting Tonality Evaluation...


✓ Tonality Score: 0.90

[4/4] Starting Safety Evaluation...


✓ Safety Score: 0.90

EVALUATION RESULTS
{
  "SummarizationScore": 0.0,
  "SummarizationReason": "",
  "CoherenceScore": 0.6,
  "CoherenceReason": "",
  "TonalityScore": 0.9,
  "TonalityReason": "",
  "SafetyScore": 0.9,
  "SafetyReason": ""
}


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [24]:
improvement_prompt = f"""
You are an expert editor and reviewer.

Your task is to improve an existing summary using evaluation feedback.

--- ORIGINAL ARTICLE ---
{article_text}

--- CURRENT SUMMARY ---
{parsed_summary.Summary}

--- EVALUATION FEEDBACK ---
Summarization Score: {summ_score}
Summarization Feedback: {summ_reason}

Coherence Score: {coh_score}
Coherence Feedback: {coh_reason}

Tonality Score: {ton_score}
Tonality Feedback: {ton_reason}

Safety Score: {safe_score}
Safety Feedback: {safe_reason}

--- INSTRUCTIONS ---
Revise the summary to:
1. Improve coverage of key points (if missing)
2. Improve clarity and logical flow
3. Strictly maintain the tone: {parsed_summary.Tone}
4. Fix any safety or factual issues
5. Keep it concise (<= 1000 tokens)

Return ONLY the improved summary.
"""

improved_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You improve summaries using evaluation feedback."},
        {"role": "user", "content": improvement_prompt}
    ],
)

improved_summary = improved_response.choices[0].message.content

improved_test_case = LLMTestCase(
    input=article_text,
    actual_output=improved_summary
)

# Reuse same metrics
improved_summarization_score = summarization_metric.measure(improved_test_case)
improved_coherence_score = coherence_metric.measure(improved_test_case)
improved_tonality_score = tonality_metric.measure(improved_test_case)
improved_safety_score = safety_metric.measure(improved_test_case)

# Extract again
impr_summ_score, _ = extract_score_and_reason(improved_summarization_score)
impr_coh_score, _ = extract_score_and_reason(improved_coherence_score)
impr_ton_score, _ = extract_score_and_reason(improved_tonality_score)
impr_safe_score, _ = extract_score_and_reason(improved_safety_score)


print("\n" + "="*70)
print("COMPARISON RESULTS")
print("="*70)

print(f"Summarization: {summ_score:.2f} → {impr_summ_score:.2f}")
print(f"Coherence:     {coh_score:.2f} → {impr_coh_score:.2f}")
print(f"Tonality:      {ton_score:.2f} → {impr_ton_score:.2f}")
print(f"Safety:        {safe_score:.2f} → {impr_safe_score:.2f}")


COMPARISON RESULTS
Summarization: 0.00 → 0.00
Coherence:     0.60 → 0.80
Tonality:      0.90 → 0.50
Safety:        0.90 → 0.90


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
